# Regional European Daily Event Analysis: Marine Heatwave ID & Tracking using `MarEx` Regional Mode

This notebook identifies and tracks the marine heatwaves in the shifting-baseline record written by
[`01_preprocess_extremes.ipynb`](01_preprocess_extremes.ipynb). It writes `events_merged.zarr` (with mergers and
splits followed, plus a log of the mergers in `events_merged_merges.nc`) and `events_basic.zarr` (basic tracking), which
[`03_visualise_events.ipynb`](03_visualise_events.ipynb) plots. The tracking pipeline is the one in
[`../gridded/02_id_track_events.ipynb`](../gridded/02_id_track_events.ipynb), scaled to the 0.05° grid
(`grid_resolution=0.05`, `R_fill=16`). Every parameter is described under
[Tracking & Merge/Split](https://marex.readthedocs.io/en/latest/guide/tracking.html#parameters).

`marEx.regional_tracker()` is `marEx.tracker()` with `regional_mode=True`. Connected components are labelled without
any wrap at the domain edges, so an object touching the 16°W edge is not joined to one at 37°E, and
`coordinate_units` is given explicitly because auto-detection is unreliable over a partial coordinate range.

Compute: one 64 GB job on the shared partition, with a local cluster of 4 workers × 16 threads × 12 GB. The table
at the end reports the wall time and memory of each stage.

## Setup

In [1]:
from getpass import getuser
from pathlib import Path

import dask
import xarray as xr

import marEx
import marEx.helper as hpc

2026-10-19 01:08:58 - marEx - INFO - [PID:3285302] - MarEx logging configured - Level: INFO, Mode: normal


In [2]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "regional"  # where 01_preprocess_extremes.ipynb wrote its output
temp_dir = scratch_dir / "marEx" / "examples" / "tmp"  # staging for compute_mode="streaming"

# A 64 GB shared job: 4 workers x 16 threads x 12 GB = 64 threads, 48 GB for the workers, the rest for the client.
# Dask spills to disk whenever the tracker's working set outgrows the workers' memory.
n_workers, threads_per_worker, memory_limit = 4, 16, "12GB"

In [3]:
out_dir.mkdir(parents=True, exist_ok=True)

# graphs of a few hundred MB are expected at this scale
dask.config.set({"distributed.admin.large-graph-warning-threshold": "1GB"})

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

2026-10-19 01:09:01 - marEx.helper.cluster - INFO - [PID:3285302] - Starting local Dask cluster with 4 workers, 16 threads each


2026-10-19 01:09:01 - marEx.helper.dask_config - INFO - [PID:3285302] - Configuring Dask for HPC environment


2026-10-19 01:09:01 - marEx.helper.dask_config - INFO - [PID:3285302] - Dask temporary directory: /scratch/<u>/<user>/marEx/dask/tmpe6qqoj8o


2026-10-19 01:09:01 - marEx.helper.dask_config - INFO - [PID:3285302] - Dask configuration completed


2026-10-19 01:09:01 - marEx.helper.cluster - INFO - [PID:3285302] - System resources: 128 physical cores, 256 logical cores, 251.6GB total memory


...

2026-10-19 01:09:01 - marEx.helper.cluster - INFO - [PID:3285302] - Starting Local cluster startup


2026-10-19 01:09:03 - marEx.helper.cluster - INFO - [PID:3285302] - Completed Local cluster startup in 1.56s


Hostname: l40004
Forward Port: l40004:8787
Dashboard Link: localhost:8787/status
2026-10-19 01:09:03 - marEx.helper.cluster - INFO - [PID:3285302] - Local cluster started successfully - Dashboard: http://127.0.0.1:8787/status


2026-10-19 01:09:03 - marEx.helper.cluster - INFO - [PID:3285302] - After cluster startup - Memory Usage - RSS: 571.3MB, Virtual: 16809.7MB, Percent: 0.2%, Available: 203405.2MB


2026-10-19 01:09:03 - marEx.helper.resources - INFO - [PID:3285302] - Cluster: 4 workers, 64 threads, memory_limit per worker [12.0] GB (48 GB total)


## 1. Identification, Tracking, & Merging

The input is chunked in time only, since the morphology and the labelling are global in space. At 25 years of daily data one int32 field is 9282 × 720 × 1060 × 4 B ≈ 28 GB, and `compute_mode="persist"`
would keep several such fields in cluster memory at once. `compute_mode="streaming"` stages them to zarr under
`temp_dir` instead, which cuts the memory pinned in the cluster. The per-object properties and the overlap graph stay in
memory and grow with the number of events. The staging directory
outlives `run()`, so the result is written first and the directory then released with `marEx.clear_staging`. See
[Memory Parameters](https://marex.readthedocs.io/en/latest/guide/tracking.html#compute-modes-and-larger-than-memory-tracking).

In [4]:
ds = xr.open_zarr(out_dir / "extremes_shifting_hobday.zarr", chunks={"time": 25, "lat": -1, "lon": -1})
ds

<xarray.Dataset> Size: 37GB
Dimensions:         (time: 9282, lat: 720, lon: 1060, dayofyear: 366)
Coordinates:
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
  * lat             (lat) float32 3kB 30.02 30.08 30.12 ... 65.88 65.93 65.97
  * lon             (lon) float32 4kB -15.98 -15.93 -15.88 ... 36.88 36.92 36.97
  * time            (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-0...
Data variables:
    dat_anomaly     (time, lat, lon) float32 28GB dask.array<chunksize=(25, 720, 1060), meta=np.ndarray>
    extreme_events  (time, lat, lon) bool 7GB dask.array<chunksize=(25, 720, 1060), meta=np.ndarray>
    mask            (lat, lon) bool 763kB dask.array<chunksize=(720, 1060), meta=np.ndarray>
    thresholds      (lat, lon, dayofyear) float32 1GB dask.array<chunksize=(720, 1060, 25), meta=np.ndarray>
Attributes: (12/13)
    max_anomaly:           21.51153564453125
    method_anomaly:        shifting_baseline
    method_extreme:        seasonal_percentile
    method_percentile:     approximate
    precision:             0.014341023763020834
    preprocessing_steps:   ['Rolling climatology using 15 years', 'Smoothed w...
    ...                    ...
    smooth_max_missing:    0.1
    tail:                  upper
    threshold_percentile:  95
    window_days:           11
    window_spatial:        5
    window_years:          15

In [5]:
with monitor.stage("Track: merging & splitting"):
    tracker = marEx.regional_tracker(
        ds.extreme_events,
        ds.mask,
        coordinate_units="degrees",  # required in regional mode, where units cannot be inferred from a partial range
        grid_resolution=0.05,  # degrees, for object areas
        area_filter_absolute=600,  # drop objects smaller than 600 cells
        R_fill=16,  # fill holes up to 16 cells in radius
        T_fill=4,  # bridge gaps of up to 4 days within one event
        allow_merging=True,  # follow events through mergers and splits, recording the mergers in `merges_ds`
        overlap_threshold=0.25,  # link objects that overlap by at least 25 % of the smaller area
        nn_partitioning=True,  # partition merged objects by nearest parent cell (False: nearest centroid, Sun et al. 2023)
        compute_mode="streaming",
        temp_dir=str(temp_dir),
    )
    extreme_events_ds, merges_ds = tracker.run(return_merges=True)
    extreme_events_ds.to_zarr(out_dir / "events_merged.zarr", mode="w")
    merges_ds.to_netcdf(out_dir / "events_merged_merges.nc", mode="w")

marEx.clear_staging(extreme_events_ds)
extreme_events_ds = xr.open_zarr(out_dir / "events_merged.zarr", chunks={})
extreme_events_ds

2026-10-19 01:09:03 - marEx.track.tracker - INFO - [PID:3285302] - Initialising MarEx tracker


2026-10-19 01:09:03 - marEx.track.tracker - INFO - [PID:3285302] - Grid type: structured


2026-10-19 01:09:03 - marEx.track.tracker - INFO - [PID:3285302] - Parameters: R_fill=16, T_fill=4, area_filter_quartile=None, area_filter_absolute=600, prefilter_min_cells=None


2026-10-19 01:09:03 - marEx.core.compute_mode - INFO - [PID:3285302] - Streaming staging directory: /scratch/<u>/<user>/marEx/examples/tmp/marex_stage_3285302_86a85634


2026-10-19 01:09:04 - marEx.track.tracker - INFO - [PID:3285302] - Calculating cell areas from grid resolution: 0.05 degrees


...

Tracking Statistics:
   Binary Hobday to Processed Area Fraction: 0.7000620329473214
   Total Object Area IDed (cells): 244465386.0
   Number of Initial Pre-Filtered Objects: 33669
   Number of Final Filtered Objects: 27344
   Area Cutoff Threshold (cells): 600
   Accepted Area Fraction: 0.9968276490480333
   Total Events Tracked: 3646
   Total Merging Events Recorded: 6381


2026-10-19 01:21:35 - marEx.track.tracker - INFO - [PID:3285302] - Completed Computing event statistics and attributes in 3.17s


2026-10-19 01:21:35 - marEx.track.tracker - INFO - [PID:3285302] - Tracking pipeline completed successfully - 3646 events identified


Track: merging & splitting: 15.3 min, peak worker memory 22.9 GB of 48 GB (largest worker 6.4 of 12.0 GB), client 1.7 GB, spilled 0.1 GB, restarts 0


<xarray.Dataset> Size: 31GB
Dimensions:       (ID: 3646, time: 9282, lat: 720, lon: 1060, component: 2,
                   sibling_ID: 14)
Coordinates:
  * ID            (ID) int32 15kB 1 2 3 4 5 6 ... 3641 3642 3643 3644 3645 3646
  * component     (component) int64 16B 0 1
  * lat           (lat) float32 3kB 30.02 30.08 30.12 ... 65.88 65.93 65.97
  * lon           (lon) float32 4kB -15.98 -15.93 -15.88 ... 36.88 36.92 36.97
  * time          (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-05-...
Dimensions without coordinates: sibling_ID
Data variables:
    ID_field      (time, lat, lon) int32 28GB dask.array<chunksize=(1, 720, 1060), meta=np.ndarray>
    area          (time, ID) float32 135MB dask.array<chunksize=(1, 3646), meta=np.ndarray>
    centroid      (component, time, ID) float32 271MB dask.array<chunksize=(1, 1, 3646), meta=np.ndarray>
    global_ID     (time, ID) int32 135MB dask.array<chunksize=(1, 3646), meta=np.ndarray>
    merge_ledger  (time, ID, sibling_ID) int32 2GB dask.array<chunksize=(1, 3646, 14), meta=np.ndarray>
    presence      (time, ID) bool 34MB dask.array<chunksize=(1, 3646), meta=np.ndarray>
    time_end      (ID) datetime64[ns] 29kB dask.array<chunksize=(3646,), meta=np.ndarray>
    time_start    (ID) datetime64[ns] 29kB dask.array<chunksize=(3646,), meta=np.ndarray>
Attributes: (12/14)
    N_events_final:              3646
    N_objects_filtered:          27344
    N_objects_prefiltered:       33669
    R_fill:                      16
    T_fill:                      4
    accepted_area_fraction:      0.9968276490480333
    ...                          ...
    area_threshold (cells):      600
    multi_parent_merges:         2562
    nn_partitioning:             1
    overlap_threshold:           0.25
    preprocessed_area_fraction:  0.7000620329473214
    total_merges:                6381

In [6]:
merges_ds

<xarray.Dataset> Size: 1MB
Dimensions:        (merge_ID: 6381, parent_idx: 14, child_idx: 14)
Dimensions without coordinates: merge_ID, parent_idx, child_idx
Data variables:
    parent_IDs     (merge_ID, parent_idx) int32 357kB 1 2 -1 -1 ... -1 -1 -1 -1
    child_IDs      (merge_ID, child_idx) int32 357kB 4 27345 -1 -1 ... -1 -1 -1
    overlap_areas  (merge_ID, parent_idx) int32 357kB 5904 2601 -1 ... -1 -1 -1
    merge_time     (merge_ID) datetime64[ns] 51kB 1997-01-02T12:00:00 ... 202...
    n_parents      (merge_ID) int8 6kB 2 2 2 2 3 3 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
    n_children     (merge_ID) int8 6kB 2 2 2 2 3 3 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
Attributes:
    fill_value:  -1

## 2. Basic Tracking for Comparison

The standard method in the literature (e.g. Scannell et al.): no temporal gap filling, no merging or splitting, and
no independent tracking of the parts of a merged event.

In [7]:
with monitor.stage("Track: basic"):
    basic_tracker = marEx.regional_tracker(
        ds.extreme_events,
        ds.mask,
        coordinate_units="degrees",  # required in regional mode, where units cannot be inferred from a partial range
        area_filter_absolute=600,
        R_fill=16,
        T_fill=0,  # no temporal gap filling
        allow_merging=False,  # touching objects share one ID for all time, before and after they touch
        compute_mode="streaming",
        temp_dir=str(temp_dir),
    )
    extreme_events_basic_ds = basic_tracker.run()
    extreme_events_basic_ds.to_zarr(out_dir / "events_basic.zarr", mode="w")

marEx.clear_staging(extreme_events_basic_ds)
extreme_events_basic_ds = xr.open_zarr(out_dir / "events_basic.zarr", chunks={})
extreme_events_basic_ds

2026-10-19 01:24:21 - marEx.track.tracker - INFO - [PID:3285302] - Initialising MarEx tracker


2026-10-19 01:24:21 - marEx.track.tracker - INFO - [PID:3285302] - Grid type: structured


2026-10-19 01:24:21 - marEx.track.tracker - INFO - [PID:3285302] - Parameters: R_fill=16, T_fill=0, area_filter_quartile=None, area_filter_absolute=600, prefilter_min_cells=None


2026-10-19 01:24:21 - marEx.core.compute_mode - INFO - [PID:3285302] - Streaming staging directory: /scratch/<u>/<user>/marEx/examples/tmp/marex_stage_3285302_f314878e


2026-10-19 01:24:21 - marEx.track.tracker - INFO - [PID:3285302] - No cell_areas provided for structured grid - using unit areas (cell counts)


...

2026-10-19 01:27:24 - marEx.track.tracker - INFO - [PID:3285302] - Starting Computing event statistics and attributes


Tracking Statistics:
   Binary Hobday to Processed Area Fraction: 0.8089643050496855
   Total Object Area IDed (cells): 211538483.0
   Number of Initial Pre-Filtered Objects: 34810
   Number of Final Filtered Objects: 26986
   Area Cutoff Threshold (cells): 600
   Accepted Area Fraction: 0.9969083450409352
   Total Events Tracked: 4331
2026-10-19 01:27:25 - marEx.track.tracker - INFO - [PID:3285302] - Completed Computing event statistics and attributes in 0.12s


2026-10-19 01:27:25 - marEx.track.tracker - INFO - [PID:3285302] - Tracking pipeline completed successfully - 4331 events identified


Track: basic: 3.5 min, peak worker memory 13.5 GB of 48 GB (largest worker 3.9 of 12.0 GB), client 1.5 GB, spilled 0.0 GB, restarts 0


<xarray.Dataset> Size: 28GB
Dimensions:   (time: 9282, lat: 720, lon: 1060)
Coordinates:
  * lat       (lat) float32 3kB 30.02 30.08 30.12 30.17 ... 65.88 65.93 65.97
  * lon       (lon) float32 4kB -15.98 -15.93 -15.88 ... 36.88 36.92 36.97
  * time      (time) datetime64[ns] 74kB 1997-01-01T12:00:00 ... 2022-05-31T1...
Data variables:
    ID_field  (time, lat, lon) int32 28GB dask.array<chunksize=(1, 720, 1060), meta=np.ndarray>
Attributes:
    N_events_final:              4331
    N_objects_filtered:          26986
    N_objects_prefiltered:       34810
    R_fill:                      16
    T_fill:                      0
    accepted_area_fraction:      0.9969083450409352
    allow_merging:               0
    area_filter_quartile:        0.0
    area_threshold (cells):      600
    preprocessed_area_fraction:  0.8089643050496855

## Wall Time and Memory

Peak memory summed over workers, the largest single worker, and the client (this kernel), per stage.

In [8]:
monitor.summary()

,wall (min),peak worker memory (GB),peak per worker (GB),peak client memory (GB),peak spilled to disk (GB),worker restarts
stage,,,,,,
Track: merging & splitting,15.27,22.87,6.39,1.73,0.06,0
Track: basic,3.48,13.47,3.90,1.54,0.00,0
total,18.75,22.87,6.39,1.73,0.06,0


In [9]:
client.close()